# Study Tutor — RAG pipeline

**AI Agentic Engineering · Corte 1**

Same phases as the Week 4 lab, run on real PDFs instead of a sample string.
Put your PDFs in `../data/` and run all cells.

The pipeline code lives in `tutor/` so the app and the CLI reuse it.
The agents built on top are in `agents.ipynb`.


## 0. Setup


In [ ]:
%pip install -q -r ../requirements.txt

In [1]:
# Pick up edits to tutor/*.py without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

# The notebook lives in notebooks/, the package one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from tutor import config   # importing this creates .env from .env.example if missing

# autoreload watches .py files, not .env - settings are re-read explicitly.
config.reload()

# Silence the SDK's automatic-function-calling notice; we use no tools.
logging.getLogger('google_genai.models').setLevel(logging.ERROR)

print('repo root      ', config.ROOT_DIR)
print('chat model     ', config.GEMINI_MODEL)
print('embed model    ', config.GEMINI_EMBED_MODEL, f'({config.EMBED_DIM} dims)')
print('chunk size     ', config.CHUNK_SIZE, 'chars, overlap', config.CHUNK_OVERLAP)
print('local fallback ', 'on' if config.ENABLE_LOCAL_FALLBACK else 'off (Gemini only)')
print('PDFs in data/  ', [p.name for p in config.DATA_DIR.glob('*.pdf')] or 'NONE - add one!')

# Fail here, not 20 cells later with a 404 that blames the model.
try:
    config.validate_models()
except Exception as error:
    print('\nCONFIGURATION PROBLEM:\n')
    print(error)

if config.GOOGLE_API_KEY:
    print('API key        loaded OK')
else:
    print(f'API key        MISSING -> open {config.ENV_FILE} and paste your key')


repo root       C:\Users\ACER\Desktop\AI_Agentic_Engineering_Project
chat model      gemini-3.6-flash
embed model     gemini-embedding-001 (768 dims)
chunk size      1200 chars, overlap 200
local fallback  on
PDFs in data/   ['NotacionYDistribucion.pdf']
API key        loaded OK


## 1. Chunking

Our own function — no text splitter library. Two rules:

- Pack whole paragraphs until the size budget runs out, so no chunk starts mid-sentence.
- Start each chunk with the tail of the previous one, so an idea explained across a
  paragraph break is findable from both sides.

These chunks are what gets indexed in §2 — this cell builds the store, it is not a
demonstration beside it.


In [7]:
import hashlib

# Reading the PDF and stripping repeated headers/footers - not chunking, and the text
# must come out byte-identical to what tutor/ingest.py produces, or the ids below would
# differ and scripts/ingest.py would store every chunk a second time.
from tutor.ingest import clean_pages, load_directory


def chunk_text(text: str, size: int = 1200, overlap: int = 200) -> list[str]:
    paragraphs = [p.strip() for p in text.split('\n') if p.strip()]

    chunks, current, length = [], [], 0
    for paragraph in paragraphs:
        # +1 for the newline that join() puts back; without it chunks run over budget.
        if current and length + len(paragraph) + 1 > size:
            chunk = '\n'.join(current)
            chunks.append(chunk)
            tail = chunk[-overlap:]              # carry the end into the next chunk
            current, length = [tail, paragraph], len(tail) + len(paragraph)
        else:
            current.append(paragraph)
            length += len(paragraph) + 1
    if current:
        chunks.append('\n'.join(current))
    return [c.strip() for c in chunks if c.strip()]


pages = clean_pages(load_directory(config.DATA_DIR))

chunks = []
for page in pages:
    for index, piece in enumerate(chunk_text(page.text, config.CHUNK_SIZE, config.CHUNK_OVERLAP)):
        # id = hash of the chunk itself, so re-running overwrites instead of duplicating
        key = f'{page.source}|{page.page_number}|{index}|{piece}'
        chunks.append({
            'id': hashlib.sha256(key.encode()).hexdigest()[:16],
            'text': piece,
            'metadata': {'source': page.source, 'page': page.page_number, 'chunk_index': index},
        })

characters = sum(len(page.text) for page in pages)
print(f'{len(pages)} pages, {characters} chars -> {len(chunks)} chunks')
print(f'sizes: min {min(len(c["text"]) for c in chunks)}, '
      f'max {max(len(c["text"]) for c in chunks)}, target {config.CHUNK_SIZE}\n')

for chunk in chunks[:2]:
    print(f"[p.{chunk['metadata']['page']}] {len(chunk['text'])} chars")
    print(f"  {chunk['text'][:100]}...\n")

print('overlap, end of chunk 0 vs start of chunk 1:')
print(' ', repr(chunks[0]['text'][-55:]))
print(' ', repr(chunks[1]['text'][:55]))


28 pages, 6759 chars -> 28 chunks
sizes: min 11, max 575, target 1200

[p.1] 16 chars
  NOTACIÓN KENDALL...

[p.2] 168 chars
  Es una herramienta fundamental en la teoría de colas porque permite
identificar y describir de maner...

overlap, end of chunk 0 vs start of chunk 1:
  'NOTACIÓN KENDALL'
  'Es una herramienta fundamental en la teoría de colas po'


The same function also lives in `tutor/ingest.py`, because notebook cells cannot be
imported and the app and the CLI need it. `tests/test_notebook_chunker.py` fails if the
two ever disagree.


---
## 2. Indexing with ChromaDB

Embed each chunk and store it with its `source` and `page`. `upsert` rather than `add`:
the ids are content hashes, so re-running this cell replaces instead of duplicating.


In [8]:
from tutor.embeddings import embed_documents
from tutor.vectorstore import get_collection

collection = get_collection()
collection.upsert(
    ids=[c['id'] for c in chunks],
    documents=[c['text'] for c in chunks],
    embeddings=embed_documents([c['text'] for c in chunks]),
    metadatas=[c['metadata'] for c in chunks],
)
print(collection.count(), 'chunks indexed')


  -> opening ChromaDB ...
  <- opening ChromaDB done in 0.2s
  cache hit on all 28 - no API call needed
28 chunks indexed


### Retrieval check

Read the scores. Below ~0.4 the chunking needs fixing.


In [9]:
from tutor.vectorstore import search

# Change this to something you know your own PDF answers.
QUESTION = 'resumen de los puntos principales del documento'

for hit in search(QUESTION, top_k=3):
    meta = hit['metadata']
    print(f"[{hit['score']:.3f}] {meta['source']} p.{meta['page']}")
    print('   ', hit['text'][:280].replace(chr(10), ' '), '...')
    print()


  cache hit on all 1 - no API call needed
  -> searching 28 chunks ...
  <- searching 28 chunks done in 0.0s
[0.672] NotacionYDistribucion.pdf p.2
    Es una herramienta fundamental en la teoría de colas porque permite identificar y describir de manera abreviada las características principales de un sistema de espera. ...

[0.672] NotacionYDistribucion.pdf p.11
    4. d.Disciplina de la fila Esta parte indica:¿Quién es atendido primero? 5. e . Capacidad máxima del sistema:¿Cuántas personas como máximo pueden estar dentro del sistema? 6. f .Tamaño de la población potencial. ¿De dónde pueden provenir los clientes? ...

[0.657] NotacionYDistribucion.pdf p.29
    REFERENCIAS Ríos, R. (25 de abril de 2021). DISTRIBUCION DE POISSON - Clase completa - Conceptos Básicos, Ejemplos y Aplicaciones [Archivo de Vídeo]. YouTube. https://www.youtube.com/watch?v=NGwiwYEpDNA Questions Of Science. (9 de febrero de 2021). Distribución Exponencial [Clase ...



### 2a. The same vectors, ranked by hand

Chroma computes cosine inside its index. `tutor/vectormath.py` computes it by hand.
If they disagree, the collection is using a metric we did not expect — and that
failure is silent.

`scripts/compare_stores.py` adds FAISS as a third opinion.


In [10]:
from tutor.vectorstore import compare_backends

check = compare_backends(QUESTION, top_k=5)
print('same ordering as Chroma :', check['same_order'])
print('largest score difference:', f"{check['max_score_difference']:.6f}")

for ours, theirs in zip(check['ours'], check['chroma']):
    print(f"  ours {ours['score']:.4f}   chroma {theirs['score']:.4f}   "
          f"p.{ours['metadata']['page']}")


  cache hit on all 1 - no API call needed
  -> searching 28 chunks ...
  <- searching 28 chunks done in 0.0s
  cache hit on all 1 - no API call needed
same ordering as Chroma : True
largest score difference: 0.000000
  ours 0.6719   chroma 0.6719   p.2
  ours 0.6718   chroma 0.6718   p.11
  ours 0.6568   chroma 0.6568   p.29
  ours 0.6432   chroma 0.6432   p.12
  ours 0.6430   chroma 0.6430   p.7


## 3. A complete `answer_with_rag` function

Retrieve, answer only from the passages, cite the pages — and refuse when the
document does not cover the question.

Two gates: nothing above 0.45 similarity means no API call at all; and the schema
forces an explicit `answered: true/false` instead of a hedge.


In [11]:
from tutor import agents

# One question the document answers, one it does not.
PROBES = [
    'resume la idea principal del documento',
    '¿cuál es el PIB de Noruega en 2019?',
]

for probe in PROBES:
    answer, hits = agents.answer_question(probe)
    best = max((h['score'] for h in hits), default=0.0)
    print(f'\nQ: {probe}')
    print(f'   best retrieval score: {best:.2f}')
    print('  ', agents.format_answer(answer).replace(chr(10), ' '))


  -> embedding 1 text(s) with gemini-embedding-001 ...
  <- embedding 1 text(s) with gemini-embedding-001 done in 0.6s
  -> searching 28 chunks ...
  <- searching 28 chunks done in 0.0s
  -> gemini-3.6-flash ...
     still waiting on gemini-3.6-flash ... 5s
     still waiting on gemini-3.6-flash ... 10s
     still waiting on gemini-3.6-flash ... 15s
     still waiting on gemini-3.6-flash ... 20s
     still waiting on gemini-3.6-flash ... 25s
     still waiting on gemini-3.6-flash ... 30s
     still waiting on gemini-3.6-flash ... 35s
  <- gemini-3.6-flash failed after 38.4s
     Gemini busy (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - retry 1/4 in 2.3s
  -> gemini-3.6-flash (attempt 2/5) ...
     still waiting on gemini-3.6-flash (attempt 2/5) ... 5s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 10s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 15s
     still waiting on gemini-3.6-flash (attempt 2/5) ... 20s
     still waiti

## 4. Exercises

**Exercise 1** — run on your own documents: that is this whole notebook.

**Exercise 2** — metadata filtering: `where=` restricts retrieval to one document.
The filter runs before the vector search, so `top_k` is filled from the matching
subset.


In [20]:
from tutor.vectorstore import list_sources, search

documents = list(list_sources())
print('indexed documents:', documents, '\n')

PROBE = 'idea principal del documento'

print('-- across everything --')
for hit in search(PROBE, top_k=3):
    meta = hit['metadata']
    print(f"  [{hit['score']:.3f}] {meta['source']} p.{meta['page']}")

if documents:
    print(f'\n-- restricted to {documents[0]} --')
    for hit in search(PROBE, top_k=3, source=documents[0]):
        meta = hit['metadata']
        print(f"  [{hit['score']:.3f}] {meta['source']} p.{meta['page']}")


indexed documents: ['NotacionYDistribucion.pdf'] 

-- across everything --
  -> embedding 1 text(s) with gemini-embedding-001 ...
  <- embedding 1 text(s) with gemini-embedding-001 done in 1.1s
  -> searching 28 chunks ...
  <- searching 28 chunks done in 0.0s
  [0.668] NotacionYDistribucion.pdf p.2
  [0.658] NotacionYDistribucion.pdf p.11
  [0.651] NotacionYDistribucion.pdf p.26

-- restricted to NotacionYDistribucion.pdf --
  cache hit on all 1 - no API call needed
  -> searching 28 chunks ...
  <- searching 28 chunks done in 0.0s
  [0.668] NotacionYDistribucion.pdf p.2
  [0.658] NotacionYDistribucion.pdf p.11
  [0.651] NotacionYDistribucion.pdf p.26


## Wrap-up

| Phase | Where |
|---|---|
| Chunking | `tutor/ingest/chunker.py` |
| Indexing | `tutor/vectorstore.py` |
| Cosine by hand | `tutor/vectormath.py` |
| `answer_with_rag` | `tutor/agents.py` → `answer_question()` |

No LangChain and no text splitters: everything is written against the Gemini SDK,
ChromaDB and numpy.


## Appendix — secret leak check

This notebook is committed with its outputs, so a stray print could put the API key
in the repo. Run after saving, before every commit.


In [ ]:
import json, re

nb_path = Path.cwd() / 'tutor.ipynb'
raw = nb_path.read_text(encoding='utf-8')

problems = []
if config.GOOGLE_API_KEY and config.GOOGLE_API_KEY in raw:
    problems.append('your GOOGLE_API_KEY appears verbatim in the saved notebook')
for match in set(re.findall(r'AIza[0-9A-Za-z_\-]{20,}', raw)):
    problems.append(f'a Google API key pattern appears: {match[:8]}...')

if problems:
    print('DO NOT COMMIT:')
    for problem in problems:
        print(' -', problem)
    print('\nClear the offending cell output (Cell > Current Outputs > Clear), save, re-run this check,')
    print('and rotate the key at aistudio.google.com if it was ever pushed.')
else:
    print('Clean: no API key found in the saved notebook. Safe to commit.')
